Step 1: Install python 3.10

Step 2: Install packages with requirements.txt

Step 3:If you want to use GPU, please install 2.6.0+cu118 as:
        pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Step 4: pip install simulated-bifurcation==1.2.0     

$$
f(\mathbf{x}) = \mathbf{x}^T Q \mathbf{x}
$$

**Where:**

- The variable \( \mathbf{x} \) is a vector in the binary domain, i.e.,
  $$
  \mathbf{x} \in \{0, 1\}^2.
  $$
- The quadratic coefficient matrix \( Q \) is given by:
  $$
  Q = \begin{pmatrix}
  1.0 & -2.0 \\
  0.0 &  3.0
  \end{pmatrix}.
  $$

This formulation corresponds to the following Python code (use GPU):




In [ ]:
import torch
import simulated_bifurcation as sb

# Define a simple 2D polynomial
Q = torch.tensor([[1.0, -2.0],
                  [0.0,  3.0]], dtype=torch.float32)

# Minimize the polynomial in a binary domain:
best_vector, best_val = sb.minimize(Q, input_type="binary",device="cuda")
print("Best vector (binary domain):", best_vector)
print("Best value:", best_val)

If you need an Ising model, please use:

In [ ]:

best_vector, best_val = sb.maximize(Q, input_type="spin")
print("Best vector (spin domain):", best_vector)
print("Best value:", best_val)

$$
\begin{aligned}
f(x,y,z) &= x^2 - y^2 + 2z^2 + xy + 2xz - 2yz - 2zx - x + 2z + 2 \\
         &= x^2 - y^2 + 2z^2 + xy - 2yz - x + 2z + 2 
         \quad \text{(since } 2xz - 2zx = 0\text{)}.
\end{aligned}
$$

We wish to represent \( f(x,y,z) \) in the quadratic form:
$$
f(x,y,z) = \begin{pmatrix} x & y & z \end{pmatrix}
\underbrace{
\begin{pmatrix}
Q_{xx} & Q_{xy} & Q_{xz} \\
Q_{xy} & Q_{yy} & Q_{yz} \\
Q_{xz} & Q_{yz} & Q_{zz}
\end{pmatrix}
}_{Q}
\begin{pmatrix} x \\ y \\ z \end{pmatrix}
+ \underbrace{\begin{pmatrix} \ell_x & \ell_y & \ell_z \end{pmatrix}}_{\boldsymbol{\ell}}
\begin{pmatrix} x \\ y \\ z \end{pmatrix}
+ c.
$$

### Extracting the Coefficients

**Quadratic Terms:**

- **\(x^2\):**  
  The coefficient is \(1\), so \(Q_{xx} = 1\).

- **\(-y^2\):**  
  The coefficient is \(-1\), so \(Q_{yy} = -1\).

- **\(2z^2\):**  
  The coefficient is \(2\), so \(Q_{zz} = 2\).

- **\(xy\):**  
  In a symmetric quadratic form, the cross term \(xy\) is represented as \(2Q_{xy}xy\).  
  Since the coefficient of \(xy\) is \(1\), we have:  
  \[
  2Q_{xy} = 1 \quad \Longrightarrow \quad Q_{xy} = 0.5.
  \]

- **\(yz\):**  
  The term is \(-2yz\). Expressing it as \(2Q_{yz}yz\) gives:  
  \[
  2Q_{yz} = -2 \quad \Longrightarrow \quad Q_{yz} = -1.
  \]

- **\(xz\):**  
  After cancellation, there is no \(xz\) term, so:  
  \[
  Q_{xz} = 0.
  \]

Thus, the symmetric matrix \( Q \) is:
$$
Q = \begin{pmatrix}
1   & 0.5 & 0   \\
0.5 & -1  & -1  \\
0   & -1  & 2
\end{pmatrix}.
$$

**Linear Terms:**

- For \(x\): the term is \(-x\), so \(\ell_x = -1\).
- For \(y\): there is no linear term, so \(\ell_y = 0\).
- For \(z\): the term is \(2z\), so \(\ell_z = 2\).

Thus, the linear vector is:
$$
\boldsymbol{\ell} = \begin{pmatrix} -1 \\ 0 \\ 2 \end{pmatrix}.
$$

**Constant Term:**

The constant term is:
$$
c = 2.
$$

### Summary

The polynomial
$$
f(x,y,z) = x^2 - y^2 + 2z^2 + xy + 2xz - 2yz - 2zx - x + 2z + 2
$$
simplifies to
$$
f(x,y,z) = x^2 - y^2 + 2z^2 + xy - 2yz - x + 2z + 2,
$$
and can be represented in the standard quadratic form as:
$$
f(x,y,z) = \begin{pmatrix} x & y & z \end{pmatrix}
\begin{pmatrix}
1 & 0.5 & 0 \\
0.5 & -1 & -1 \\
0 & -1 & 2
\end{pmatrix}
\begin{pmatrix} x \\ y \\ z \end{pmatrix}
+ \begin{pmatrix} -1 & 0 & 2 \end{pmatrix}
\begin{pmatrix} x \\ y \\ z \end{pmatrix}
+ 2.
$$


In [ ]:
# Define the numerical matrix Q, vector l, and constant c
Q = torch.tensor([
    [1.0,  0.5,  0.0],
    [0.5, -1.0, -1.0],
    [0.0, -1.0,  2.0]
], dtype=torch.float32)

l = torch.tensor([-1.0, 0.0, 2.0], dtype=torch.float32)
c = 2.0

# Build the polynomial model for the spin domain (values in {-1, 1})
poly_model = sb.build_model(Q, vector=l, constant=c, input_type="binary")

# Run the optimization (minimize in this case)
all_vectors, all_values = poly_model.minimize(agents=64, best_only=False)

print("All solutions shape:", all_vectors.shape)
print("All evaluations shape:", all_values.shape)

The upper script run the model 64 times and show all result